In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS bfsi.bronze_layer
    COMMENT 'Data Ingestion - custom schema and data partitioning'
""")
print("Bronze Layer Created..")

In [0]:
display(spark.sql("SHOW VOLUMES IN bfsi.landing"))

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOG         = "bfsi"
SCHEMA          = "bronze_layer"
SOURCE_PATH     = "/Volumes/bfsi/landing/card/"
TABLE_NAME      = f"{CATALOG}.{SCHEMA}.card_transactions"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/card/_checkpoint/{SCHEMA}_card_transactions"
SOURCE_SYSTEM   = "CARD"
BATCH_ID        = dbutils.widgets.get("batch_id") if "batch_id" in [w.name for w in dbutils.widgets.getAll()] else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

card_schema = StructType([
    StructField("txn_id",                 StringType(),  True),
    StructField("card_number",            StringType(),  True),
    StructField("customer_id",            StringType(),  True),
    StructField("merchant_id",            StringType(),  True),
    StructField("merchant_name",          StringType(),  True),
    StructField("merchant_category_code", StringType(),  True),
    StructField("txn_amount",             DoubleType(),  True),
    StructField("txn_currency",           StringType(),  True),
    StructField("txn_ts",                 StringType(),  True),
    StructField("txn_type",               StringType(),  True),
    StructField("pos_entry_mode",         StringType(),  True),
    StructField("country_code",           StringType(),  True),
    StructField("is_international",       BooleanType(), True),
    StructField("auth_code",              StringType(),  True),
    StructField("response_code",          StringType(),  True),
])

df_card_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.includeExistingFiles", "true")
    .schema(card_schema)
    .load(SOURCE_PATH)
)

print("Auto Loader stream configured for card transactions")

In [0]:
df_card_bronze = (
    df_card_raw
    .withColumn("txn_ts", F.to_timestamp("txn_ts"))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
)

(
    df_card_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)


print(f"Streaming write to {TABLE_NAME} started")

In [0]:
df_verify = spark.read.table(TABLE_NAME)
print(f"Total records in {TABLE_NAME}: {df_verify.count()}")
display(df_verify.count())